# Estabilidade e rigidez

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/13-estabilidade-rigidez.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. desenhar e usar a região de estabilidade de um método para escolher o passo;
> 2. reconhecer um problema **rígido** pelos sintomas e pelos autovalores da jacobiana;
> 3. explicar por que métodos implícitos (Euler implícito, trapézio, BDF, Radau) resolvem problemas rígidos, e o que eles custam;
> 4. usar integradores simpléticos em simulações longas de sistemas conservativos;
> 5. entender o que se pode e o que não se pode prever num sistema caótico.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
import time
from scipy.integrate import solve_ivp
from cnlivro.edo import passo_fixo, implicito, verlet

## 1. Problema de engenharia: a temperatura de um módulo de potência

O inversor que alimenta o motor de um ônibus elétrico usa módulos de potência (IGBTs). Cada vez que o ônibus acelera, o chip dissipa centenas de watts e esquenta em milissegundos; o calor passa para a base de cobre e depois para o dissipador, que leva minutos para aquecer. A vida útil do módulo depende de duas grandezas: a temperatura máxima da junção, $T_j$, e a **amplitude** $\Delta T_j$ com que ela oscila a cada ciclo, porque a fadiga das soldas cresce muito depressa com $\Delta T_j$.

Um modelo térmico comum é uma rede de resistências e capacitâncias térmicas, com um nó para a junção ($j$), um para a base ($c$) e um para o dissipador ($d$):

$$
\begin{aligned}
C_j\,\dot T_j &= P(t) - \frac{T_j - T_c}{R_{jc}},\\
C_c\,\dot T_c &= \frac{T_j - T_c}{R_{jc}} - \frac{T_c - T_d}{R_{cd}},\\
C_d\,\dot T_d &= \frac{T_c - T_d}{R_{cd}} - \frac{T_d - T_a}{R_{da}},
\end{aligned}
$$ (eq-modulo)

com $P(t)$ a potência dissipada. O ciclo de operação repete 300 W durante 20 s (aceleração) e 30 W durante 40 s (velocidade de cruzeiro). Queremos $T_j$ máxima e $\Delta T_j$ depois de uma hora de operação. É um sistema **linear** de três EDOs, $\dot{\mathbf T} = A\mathbf T + \mathbf b(t)$: nada mais simples. Mesmo assim, ele vai dar trabalho.

In [ ]:
Cj, Cc, Cd = 0.05, 30.0, 1500.0          # capacitâncias térmicas, J/K
Rjc, Rcd, Rda = 0.2, 0.05, 0.12          # resistências térmicas, K/W
Ta = 40.0                                # temperatura ambiente, °C

A = np.array([[-1 / (Rjc * Cj), 1 / (Rjc * Cj), 0.0],
              [1 / (Rjc * Cc), -(1 / Rjc + 1 / Rcd) / Cc, 1 / (Rcd * Cc)],
              [0.0, 1 / (Rcd * Cd), -(1 / Rcd + 1 / Rda) / Cd]])

def potencia(t):
    return np.where(t % 60 < 20, 300.0, 30.0)            # W

def modulo(t, T):
    return A @ T + np.array([potencia(t) / Cj, 0.0, Ta / (Rda * Cd)])

T_ini = np.full(3, Ta)
lam = np.linalg.eigvals(A).real
print("autovalores de A (1/s):", np.array2string(np.sort(lam), precision=4))
print("constantes de tempo (s):", np.array2string(np.sort(-1 / lam), precision=4))
print(f"razão entre a mais lenta e a mais rápida: {lam.min() / lam.max():.0f}")

As três constantes de tempo vão de 10 ms (o chip) a 3 minutos (o dissipador): uma razão de quase 20 mil. Como o sistema é linear e a potência é constante por trechos, há solução **exata**: em cada trecho, $\mathbf T(t) = \mathbf T_{\text{eq}} + e^{A(t - t_k)}\big(\mathbf T(t_k) - \mathbf T_{\text{eq}}\big)$, com a exponencial de matriz calculada pelos autovalores e autovetores de $A$ (capítulo 8). Ela será a nossa referência.

In [ ]:
autoval, V = np.linalg.eig(A)
V_inv = np.linalg.inv(V)

def exata_modulo(t_fim, pontos_por_trecho=400):
    # solução exata trecho a trecho; devolve instantes e temperaturas (linhas: Tj, Tc, Td)
    ts, Ts = [0.0], [T_ini.copy()]
    t, T = 0.0, T_ini.copy()
    while t < t_fim - 1e-9:
        dur = 20.0 if t % 60 < 20 - 1e-9 else 40.0
        P = 300.0 if dur == 20.0 else 30.0
        T_eq = -np.linalg.solve(A, np.array([P / Cj, 0.0, Ta / (Rda * Cd)]))
        tau = np.r_[np.geomspace(1e-4, 0.2, pontos_por_trecho // 2), np.linspace(0.2, dur, pontos_por_trecho // 2)[1:]]
        trecho = T_eq[:, None] + (V @ (np.exp(np.outer(autoval, tau)) * (V_inv @ (T - T_eq))[:, None])).real
        ts.extend(t + tau)
        Ts.extend(trecho.T)
        t, T = t + dur, trecho[:, -1]
    return np.array(ts), np.array(Ts).T

t_ex, T_ex = exata_modulo(3600)
ultimo = t_ex >= 3540
print(f"Tj máxima em 1 h: {T_ex[0].max():.2f} °C;  ΔTj no último ciclo: {np.ptp(T_ex[0][ultimo]):.2f} K")
print(f"temperatura do dissipador no fim: {T_ex[2, -1]:.2f} °C")

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=["a primeira hora", "um ciclo, 58 min depois"])
nomes = ["junção Tj", "base Tc", "dissipador Td"]
cores = [CORES["erro"], CORES["terciaria"], CORES["aproximacao"]]
passo = 7
for i in range(3):
    fig.add_trace(go.Scatter(x=t_ex[::passo] / 60, y=T_ex[i, ::passo], mode="lines", name=nomes[i],
                             line=dict(color=cores[i], width=1)), 1, 1)
    m = (t_ex >= 3480) & (t_ex <= 3540)
    fig.add_trace(go.Scatter(x=t_ex[m] - 3480, y=T_ex[i, m], mode="lines", showlegend=False,
                             line=dict(color=cores[i], width=2)), 1, 2)
fig.update_xaxes(title="t (min)", row=1, col=1)
fig.update_xaxes(title="t (s), dentro do ciclo", row=1, col=2)
fig.update_yaxes(title="temperatura (°C)", row=1, col=1)
fig.update_layout(width=980, height=430, legend=dict(orientation="h", y=-0.22))
mostrar(fig)

**O que notar:** o dissipador sobe devagar ao longo de toda a hora, enquanto a junção salta dezenas de graus em milissegundos a cada troca de carga. À direita, num ciclo, $T_j$ vai de cerca de 60 °C a 130 °C. A solução tem uma parte lenta e suave e uma parte rápida que só importa logo depois de cada troca.

## 2. A ideia em uma imagem

Para entender o que vai acontecer, olhe a equação mais simples com uma constante de tempo: $y' = \lambda y$, com $\lambda = -10$ s⁻¹ e $y(0) = 1$. A solução, $e^{-10t}$, decai suavemente. Arraste o slider para aumentar o passo $h$.

In [ ]:
lam2 = -10.0
tt = np.linspace(0, 2, 300)

def quadro_estab(h):
    n = int(round(2 / h))
    tn = h * np.arange(n + 1)
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=["Euler explícito", "Euler implícito"])
    for col, fator in [(1, 1 + h * lam2), (2, 1 / (1 - h * lam2))]:
        fig.add_trace(go.Scatter(x=tt, y=np.exp(lam2 * tt), mode="lines", line=dict(color=CORES["exata"], width=2.5),
                                 name="exata", showlegend=(col == 1)), 1, col)
        fig.add_trace(go.Scatter(x=tn, y=fator ** np.arange(n + 1), mode="lines+markers", name="numérica",
                                 line=dict(color=CORES["erro"] if col == 1 else CORES["aproximacao"]),
                                 showlegend=False), 1, col)
    fig.update_xaxes(title="t (s)")
    fig.update_yaxes(range=[-2.2, 2.2])
    fig.update_layout(width=980, height=430, showlegend=False, margin=dict(t=110),
                      title=f"h = {numero(h)} s, hλ = {numero(h * lam2)}: fator por passo {numero(1 + h * lam2, 3)} "
                            f"(explícito) e {numero(1 / (1 - h * lam2), 3)} (implícito)")
    return fig

hs_q = [0.05, 0.15, 0.19, 0.21, 0.3]
quadros = [(numero(h), quadro_estab(h)) for h in hs_q]
mostrar(com_controle(quadros, rotulo="h = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** cada passo de Euler explícito multiplica $y$ por $1 + h\lambda$. Com $h = 0{,}15$ s, o fator é $-0{,}5$: a solução troca de sinal a cada passo, mas decai. Com $h = 0{,}21$ s, o fator é $-1{,}1$ e a solução **explode**, embora a exata vá a zero. A fronteira é $h\lambda = -2$. O Euler implícito multiplica por $1/(1 - h\lambda)$, que é menor que 1 para **qualquer** $h > 0$: ele nunca explode, por maior que seja o passo.

No módulo de potência, o chip tem $\lambda \approx -100$ s⁻¹: um método explícito precisa de $h < 0{,}02$ s **durante a hora inteira**, mesmo nos longos trechos em que o chip já está em equilíbrio e só o dissipador, com constante de tempo de 3 minutos, ainda muda. É isso que se chama de **rigidez**.

## 3. Formulação matemática

### 3.1 A equação teste e a região de estabilidade

Aplicado a $y' = \lambda y$, com $\lambda$ complexo, todo método de passo único dá $y_{n+1} = R(z)\,y_n$, com $z = h\lambda$. A função $R$ é o **fator de amplificação**:

| Método | $R(z)$ |
| --- | --- |
| Euler explícito | $1 + z$ |
| Heun (RK2) | $1 + z + z^2/2$ |
| RK4 | $1 + z + z^2/2 + z^3/6 + z^4/24$ |
| Euler implícito | $1/(1 - z)$ |
| Trapézio (Crank-Nicolson) | $(1 + z/2)/(1 - z/2)$ |

A **região de estabilidade** é o conjunto dos $z$ com $|R(z)| \le 1$: ali, a solução numérica não cresce. Para um método explícito, $R$ é um polinômio, que cresce para $|z|$ grande; a região é **limitada**, e o passo também. Para Euler explícito, ela é o disco de raio 1 centrado em $-1$.

**Para sistemas**, $\mathbf y' = A\mathbf y$, a mudança de variável pelos autovetores de $A$ separa o sistema em equações independentes $w_i' = \lambda_i w_i$ (capítulo 8). O método é estável se $h\lambda_i$ está na região de estabilidade **para todo** $i$. Num sistema não linear, o papel de $A$ é feito pela **jacobiana** $\partial \mathbf f/\partial \mathbf y$ ao longo da solução.

### 3.2 Rigidez

Um problema é **rígido** quando a jacobiana tem autovalores com parte real muito negativa, que correspondem a modos que decaem depressa e logo deixam de importar, enquanto a solução que interessa varia devagar. Para um método explícito, o passo fica preso a $h \lesssim c/|\lambda_{\max}|$ pela **estabilidade**, embora a **precisão** permitisse um passo muito maior. A razão $\lambda_{\max}/\lambda_{\min}$ mede a rigidez: no módulo, quase 20 mil.

Os sintomas práticos são inconfundíveis:

- um método explícito adaptativo dá muitos passos, de tamanho quase constante;
- **afrouxar a tolerância não diminui o número de passos** (o passo é limitado pela estabilidade, não pela precisão);
- com passo fixo grande, a solução explode ou oscila.

Problemas rígidos aparecem sempre que há escalas de tempo muito diferentes: cinética química com reações rápidas e lentas, circuitos com capacitâncias parasitas, estruturas com modos de alta frequência, e a discretização de EDPs de difusão (capítulos 15 e 16).

### 3.3 Métodos implícitos

O **Euler implícito** usa a inclinação no **fim** do passo:

$$
\mathbf y_{n+1} = \mathbf y_n + h\,\mathbf f(t_{n+1}, \mathbf y_{n+1}) .
$$ (eq-euler-implicito)

A incógnita $\mathbf y_{n+1}$ aparece dos dois lados: cada passo exige resolver um sistema de equações, em geral não linear, pelo método de Newton (capítulo 5). Com $J$ a jacobiana de $\mathbf f$, cada iteração resolve o sistema linear $(I - hJ)\,\Delta = -\mathbf G$ (capítulo 4). Um passo implícito custa muito mais que um explícito, mas o passo pode ser milhares de vezes maior.

Dois conceitos classificam os implícitos:

- **A-estável:** a região de estabilidade contém todo o semiplano $\operatorname{Re} z < 0$. Euler implícito e trapézio são A-estáveis.
- **L-estável:** além disso, $R(z) \to 0$ quando $z \to -\infty$, ou seja, modos muito rápidos são **amortecidos** num passo. Euler implícito é L-estável ($R \to 0$); o trapézio não é ($R \to -1$), e por isso deixa oscilar os modos rápidos (seção 7.2).

### 3.4 Métodos de passo múltiplo

Métodos de **passo múltiplo** reaproveitam valores de passos anteriores. **Adams-Bashforth** (explícitos) integram um polinômio que interpola os $\mathbf f$ anteriores; o de dois passos é

$$
\mathbf y_{n+1} = \mathbf y_n + \frac h2\left(3\mathbf f_n - \mathbf f_{n-1}\right),
$$

de ordem 2, com uma avaliação de $\mathbf f$ por passo. As fórmulas **BDF** (*backward differentiation formulas*) são implícitas: derivam o polinômio que interpola $\mathbf y_{n+1}, \mathbf y_n, \dots$ no instante $t_{n+1}$. A de ordem 2 é

$$
\mathbf y_{n+1} - \tfrac43\mathbf y_n + \tfrac13\mathbf y_{n-1} = \tfrac23\,h\,\mathbf f(t_{n+1}, \mathbf y_{n+1}).
$$

`solve_ivp(method="BDF")` varia a ordem de 1 a 5 e o passo. `Radau` é um Runge-Kutta implícito de ordem 5, L-estável. `LSODA` começa com um Adams explícito e troca para BDF quando detecta rigidez. Há um limite teórico, a **barreira de Dahlquist**: nenhum método de passo múltiplo A-estável tem ordem maior que 2. As BDF de ordem 3 a 5 são estáveis em quase todo o semiplano esquerdo, mas não perto do eixo imaginário, onde ficam os modos oscilatórios pouco amortecidos.

### 3.5 Integradores simpléticos

Em sistemas **conservativos** (órbitas, pêndulos, dinâmica molecular), não há modo que decaia: os autovalores ficam no eixo imaginário. Um método cuja região não contém o eixo imaginário (Euler explícito) faz a energia crescer; um que é estável demais (Euler implícito) faz a energia cair. Nos dois casos, uma simulação longa é inútil.

Para $\ddot{\mathbf q} = \mathbf a(\mathbf q)$, o método de **Verlet** (de velocidade) é

$$
\mathbf v_{n+1/2} = \mathbf v_n + \tfrac h2\,\mathbf a(\mathbf q_n), \qquad
\mathbf q_{n+1} = \mathbf q_n + h\,\mathbf v_{n+1/2}, \qquad
\mathbf v_{n+1} = \mathbf v_{n+1/2} + \tfrac h2\,\mathbf a(\mathbf q_{n+1}).
$$ (eq-verlet)

É de ordem 2, com uma avaliação de $\mathbf a$ por passo, e é **simplético**: preserva a área no plano de fase, como o sistema exato. A consequência prática é que o erro de energia **oscila sem crescer**, por milhões de passos. Simuladores de dinâmica molecular e de mecânica celeste usam integradores desse tipo.

### 3.6 Caos: o limite da previsão

Num sistema **caótico**, duas trajetórias que começam a uma distância $\delta_0$ se afastam exponencialmente, $\delta(t) \approx \delta_0\,e^{\Lambda t}$, com $\Lambda > 0$ o **expoente de Lyapunov**. Qualquer erro, de medida ou de arredondamento, cresce assim. A trajetória só é previsível até um **horizonte**

$$
t_h \approx \frac{1}{\Lambda}\ln\frac{\Delta}{\delta_0},
$$

com $\Delta$ o tamanho do erro tolerável. Diminuir $\delta_0$ por um fator de mil aumenta o horizonte só em $\ln(1000)/\Lambda$. Nenhum método numérico muda isso: além do horizonte, só as **estatísticas** (médias, distribuições) são confiáveis.

## 4. À mão

Para $y' = -50\,y$, $y(0) = 1$, com $h = 0{,}1$, ou seja, $z = h\lambda = -5$ (a exata é $y(0{,}1) = e^{-5} = 0{,}0067$ e $y(0{,}2) = 0{,}000045$):

| Método | $R(-5)$ | $y_1$ | $y_2$ |
| --- | --- | --- | --- |
| Euler explícito | $1 - 5 = -4$ | $-4$ | $16$ |
| Euler implícito | $1/(1 + 5) = 0{,}1667$ | $0{,}1667$ | $0{,}0278$ |
| Trapézio | $(1 - 2{,}5)/(1 + 2{,}5) = -0{,}4286$ | $-0{,}4286$ | $0{,}1837$ |

O explícito explode. O implícito decai, embora mais devagar que a solução exata: o passo é grande demais para ser **preciso**, mas o método é **estável**. O trapézio também decai, mas trocando de sinal: é estável, mas não amortece direito um modo tão rápido.

Para obter $y_1$ no Euler implícito, resolve-se $y_1 = y_0 + h(-50\,y_1)$, isto é, $y_1(1 + 5) = 1$. Num problema não linear, essa equação se resolve por Newton.

In [ ]:
f50 = lambda t, y: -50 * y
for metodo in ["euler", "trapezio"]:
    r = implicito(f50, (0, 0.2), [1.0], 0.1, metodo)
    print(f"{metodo:8s} implícito: y1 = {r.y[1, 0]:+.4f}, y2 = {r.y[2, 0]:+.4f}  (iterações de Newton: {r.historico})")
print("Euler explícito:", passo_fixo(f50, (0, 0.2), 1.0, 0.1, "euler").y[1:, 0])

## 5. Explore

### 5.1 Regiões de estabilidade e os autovalores do módulo

As curvas abaixo são as fronteiras $|R(z)| = 1$. Os métodos explícitos são estáveis **dentro** das curvas fechadas; o Euler implícito é estável **fora** do seu círculo, e o trapézio em todo o semiplano esquerdo. Arraste o slider para mudar o passo $h$: os pontos são $h\lambda_i$ para os três autovalores do módulo.

In [ ]:
# fronteira |R(z)| = 1: para cada ângulo θ, as raízes de R(z) = e^{iθ} (polinômio em z)
theta = np.linspace(0, 2 * np.pi, 1500)
coefs = {"Euler explícito": [1, 1], "Heun (RK2)": [1 / 2, 1, 1], "RK4": [1 / 24, 1 / 6, 1 / 2, 1, 1]}
cores_reg = {"Euler explícito": CORES["erro"], "Heun (RK2)": CORES["secundaria"], "RK4": CORES["aproximacao"],
             "Euler implícito": CORES["destaque"]}
fronteiras = {}
for nome, c in coefs.items():
    z = np.concatenate([np.roots(np.r_[c[:-1], c[-1] - np.exp(1j * th)]) for th in theta])
    fronteiras[nome] = z
fronteiras["Euler implícito"] = 1 - np.exp(-1j * theta)        # 1/(1 - z) = e^{iθ}
fatores = {"Euler explícito": lambda z: 1 + z, "Heun (RK2)": lambda z: 1 + z + z**2 / 2,
           "RK4": lambda z: 1 + z + z**2 / 2 + z**3 / 6 + z**4 / 24, "Euler implícito": lambda z: 1 / (1 - z)}

def quadro_regiao(h):
    fig = go.Figure()
    for nome, z in fronteiras.items():
        fig.add_trace(go.Scatter(x=z.real, y=z.imag, mode="markers", marker=dict(color=cores_reg[nome], size=3),
                                 name=nome, hoverinfo="skip"))
    hz = h * lam
    fig.add_trace(go.Scatter(x=hz, y=np.zeros(3), mode="markers", name="hλ dos três modos",
                             marker=dict(color=CORES["exata"], size=12, symbol="x")))
    fig.add_vline(x=0, line=dict(color=CORES["referencia"], width=1))
    fig.add_hline(y=0, line=dict(color=CORES["referencia"], width=1))
    estaveis = [nome for nome, R in fatores.items() if np.all(np.abs(R(hz)) <= 1)]
    fig.update_xaxes(title="Re(hλ)", range=[-4.2, 2.2])
    fig.update_yaxes(title="Im(hλ)", range=[-3.5, 3.5], scaleanchor="x")
    fig.update_layout(width=800, height=560, legend=dict(x=1.02, y=1),
                      title=f"h = {numero(h)} s: estáveis: {', '.join(estaveis)}")
    return fig

quadros = [(numero(h), quadro_regiao(h)) for h in [0.005, 0.015, 0.025, 0.035]]
mostrar(com_controle(quadros, rotulo="h = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** os dois modos lentos ficam colados na origem para qualquer passo razoável. Só o modo do chip se move: com $h = 0{,}025$ s ele já saiu das regiões de Euler e de Heun, e com $h = 0{,}035$ s saiu também da do RK4, cuja fronteira no eixo real está em $-2{,}79$. O Euler implícito continua estável. A região do RK45 do `solve_ivp` é parecida com a do RK4, um pouco maior: no eixo real, ela vai até cerca de $-3{,}3$.

### 5.2 Energia em simulações longas: Verlet contra RK4

Um pêndulo de 1 m, solto de 1 rad, durante cerca de mil oscilações. Para comparar com o **mesmo custo**, o RK4 (quatro avaliações por passo) usa um passo quatro vezes maior que o Verlet (uma avaliação por passo).

In [ ]:
g_L = 9.81
acel_pend = lambda q: -g_L * np.sin(q)
energia_pend = lambda th, om: 0.5 * om**2 + g_L * (1 - np.cos(th))
t_fim, h_v = 2000.0, 0.05
rv = verlet(acel_pend, (0, t_fim), [1.0], [0.0], h_v)
rr = passo_fixo(lambda t, u: np.array([u[1], -g_L * np.sin(u[0])]), (0, t_fim), [1.0, 0.0], 4 * h_v, "rk4")
E0 = energia_pend(1.0, 0.0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=rr.t[::5], y=(energia_pend(rr.y[::5, 0], rr.y[::5, 1]) - E0) / E0, mode="lines",
                         name=f"RK4, h = {numero(4 * h_v)} s ({rr.avaliacoes} avaliações)", line=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=rv.t[::20], y=(energia_pend(rv.y[::20, 0], rv.y[::20, 1]) - E0) / E0, mode="lines",
                         name=f"Verlet, h = {numero(h_v)} s ({rv.avaliacoes} avaliações)",
                         line=dict(color=CORES["aproximacao"], width=1)))
fig.update_xaxes(title="t (s)")
fig.update_yaxes(title="erro relativo da energia")
fig.update_layout(width=820, height=430, title="Pêndulo: energia ao longo de cerca de mil oscilações",
                  legend=dict(x=0.02, y=0.05))
mostrar(fig)

**O que notar:** com o mesmo número de avaliações, o RK4, de ordem mais alta, perde energia de forma sistemática, um pouco a cada passo: o pêndulo numérico vai parando, como se tivesse atrito, e ao fim está quase em repouso. O erro do Verlet, de ordem 2, é de alguns milésimos e **oscila sem crescer**. Para simular por muito tempo um sistema sem atrito, a **estrutura** do método importa mais que a ordem.

### 5.3 Caos: o horizonte de previsão

O sistema de Lorenz, $\dot x = 10(y - x)$, $\dot y = x(28 - z) - y$, $\dot z = xy - \tfrac83 z$, nasceu como um modelo extremamente simplificado da convecção na atmosfera. Duas simulações partem de pontos que diferem por $\delta_0$ na coordenada $x$. Arraste o slider para mudar $\delta_0$.

In [ ]:
def lorenz(t, u):
    x, y, z = u
    return [10 * (y - x), x * (28 - z) - y, x * y - 8 / 3 * z]

u_l = [1.0, 1.0, 20.0]
tt = np.linspace(0, 40, 4001)
base = solve_ivp(lorenz, (0, 40), u_l, method="DOP853", rtol=1e-13, atol=1e-13, dense_output=True).sol(tt)

def quadro_lorenz(d0):
    outra = solve_ivp(lorenz, (0, 40), [u_l[0] + d0, u_l[1], u_l[2]], method="DOP853", rtol=1e-13, atol=1e-13,
                      dense_output=True).sol(tt)
    dist = np.linalg.norm(outra - base, axis=0)
    t_h = tt[np.argmax(dist > 1.0)]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.55, 0.45])
    fig.add_trace(go.Scatter(x=tt, y=base[0], mode="lines", name="x(t), partida A", line=dict(color=CORES["exata"], width=1.5)), 1, 1)
    fig.add_trace(go.Scatter(x=tt, y=outra[0], mode="lines", name="x(t), partida B", line=dict(color=CORES["erro"], width=1.5)), 1, 1)
    fig.add_trace(go.Scatter(x=tt, y=np.maximum(dist, 1e-16), mode="lines", name="distância entre as duas",
                             line=dict(color=CORES["aproximacao"])), 2, 1)
    fig.add_trace(go.Scatter(x=[t_h, t_h], y=[1e-16, 1e2], mode="lines", name="distância = 1",
                             line=dict(color=CORES["referencia"], dash="dash")), 2, 1)
    fig.update_yaxes(title="x", row=1, col=1)
    fig.update_yaxes(title="distância", type="log", range=[-15, 2], exponentformat="power", row=2, col=1)
    fig.update_xaxes(title="t", row=2, col=1)
    fig.update_layout(width=900, height=560, legend=dict(orientation="h", y=-0.15),
                      title=f"δ₀ = {d0:.0e}: as trajetórias se separam (distância > 1) em t ≈ {numero(t_h, 1)}"
                      .replace("e-0", "e-"))
    return fig

quadros = [(f"{d:.0e}".replace("e-0", "e-"), quadro_lorenz(d)) for d in [1e-3, 1e-6, 1e-9, 1e-12]]
mostrar(com_controle(quadros, rotulo="δ₀ = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** em escala logarítmica, a distância cresce como uma reta: crescimento exponencial, com inclinação dada pelo expoente de Lyapunov (cerca de 0,9). Cada fator mil em $\delta_0$ compra só cerca de 7 unidades de tempo de previsão. Com $\delta_0 = 10^{-12}$ o ganho é menor, porque o erro do próprio integrador, com tolerância $10^{-13}$, já é da ordem de $\delta_0$. Como a tolerância do integrador age como um $\delta_0$ a cada passo, o mesmo vale para o erro numérico: nenhuma tolerância permite prever $x(t)$ em $t = 100$.

Mas as duas trajetórias continuam no mesmo "atrator", com a mesma faixa de valores e a mesma alternância entre os dois lóbulos. Médias e distribuições de longo prazo são previsíveis; a trajetória individual não. É a diferença entre a previsão do tempo para depois de amanhã e o clima do próximo verão.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Simule uma hora do ciclo de carga do módulo de potência com `solve_ivp` e informe a temperatura máxima da junção e a amplitude $\Delta T_j$ no último ciclo."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
from scipy.integrate import solve_ivp

def rhs(t, T):
    P = 300.0 if (t % 60) < 20 else 30.0
    Tj, Tc, Td = T
    dTj = (P - (Tj - Tc) / Rjc) / Cj
    dTc = ((Tj - Tc) / Rjc - (Tc - Td) / Rcd) / Cc
    dTd = ((Tc - Td) / Rcd - (Td - Ta) / Rda) / Cd
    return [dTj, dTc, dTd]

inicio = time.perf_counter()
sol = solve_ivp(rhs, (0, 3600), [Ta, Ta, Ta])
duracao = time.perf_counter() - inicio
Tj = sol.y[0]
ult = sol.t >= 3540
print(f"Tj máxima: {Tj.max():.1f} °C")
print(f"ΔTj no último ciclo: {Tj[ult].max() - Tj[ult].min():.1f} K")
print(f"{sol.t.size - 1} passos, {sol.nfev} avaliações, {duracao:.1f} s de computação")

O modelo está transcrito corretamente e o código roda sem erros. Mas uma hora de operação levou segundos para simular. Uma análise de vida útil precisa de um ano de perfil de uso, e um projeto de dissipador, de centenas dessas análises.

### 6.2 Verificação

**Contra a solução exata** da seção 1:

In [ ]:
print(f"Tj máxima: agente {Tj.max():.2f} °C, exata {T_ex[0].max():.2f} °C")
print(f"ΔTj no último ciclo: agente {Tj[ult].max() - Tj[ult].min():.2f} K, exata {np.ptp(T_ex[0][ultimo]):.2f} K")
i = np.argmax(Tj)
print(f"o máximo do agente acontece em t = {sol.t[i]:.4f} s; vizinhos: {np.round(Tj[i - 2:i + 3], 1)}")

A amplitude no último ciclo está certa, mas a temperatura máxima está 12 K acima da exata, e o "máximo" é um **ponto isolado**, logo depois de uma troca de carga, cercado por valores muito menores. Não é física: é um artefato numérico.

**Pelos sintomas de rigidez.** Se o passo fosse limitado pela precisão, uma tolerância mais frouxa reduziria o número de passos:

In [ ]:
for rtol in [1e-2, 1e-3, 1e-6]:
    s = solve_ivp(rhs, (0, 600), [Ta, Ta, Ta], rtol=rtol)
    print(f"10 min com rtol = {rtol:.0e}: {s.t.size - 1:6d} passos, passo mediano = {np.median(np.diff(s.t)):.4f} s")
print(f"limite de estabilidade do RK45 no eixo real: h ≈ 3,3/|λ_max| = {3.3 / abs(lam.min()):.4f} s")

In [ ]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=["tamanho do passo do RK45", "Tj perto do falso máximo"])
fig.add_trace(go.Scatter(x=sol.t[1:][::20] / 60, y=np.diff(sol.t)[::20], mode="markers", name="passo do RK45",
                         marker=dict(color=CORES["erro"], size=3)), 1, 1)
fig.add_trace(go.Scatter(x=[0, 60], y=[3.3 / abs(lam.min())] * 2, mode="lines", name="3,3/|λ_max| (estabilidade)",
                         line=dict(color=CORES["exata"], dash="dash")), 1, 1)
t_m = sol.t[i]
m = (sol.t > t_m - 0.3) & (sol.t < t_m + 0.3)
me = (t_ex > t_m - 0.3) & (t_ex < t_m + 0.3)
fig.add_trace(go.Scatter(x=t_ex[me] - t_m, y=T_ex[0, me], mode="lines", name="exata",
                         line=dict(color=CORES["exata"], width=2.5)), 1, 2)
fig.add_trace(go.Scatter(x=sol.t[m] - t_m, y=Tj[m], mode="lines+markers", name="agente (RK45)",
                         line=dict(color=CORES["erro"])), 1, 2)
fig.update_xaxes(title="t (min)", row=1, col=1)
fig.update_yaxes(title="passo (s)", range=[0, 0.05], row=1, col=1)
fig.update_xaxes(title="t − t_max do agente (s)", row=1, col=2)
fig.update_yaxes(title="Tj (°C)", row=1, col=2)
fig.update_layout(width=980, height=430, legend=dict(orientation="h", y=-0.22))
mostrar(fig)

O número de passos praticamente não muda com a tolerância, e o passo fica grudado no limite de estabilidade do modo do chip durante a hora inteira. É o diagnóstico de rigidez da seção 3.2. À direita, o falso máximo: o passo que atravessou a queda de potência terminou com o modo rápido excitado, perto da fronteira de estabilidade, e o estimador de erro aceitou o resultado.

**A segunda tentativa.** Diante disso, um agente costuma propor trocar o método por um implícito:

In [ ]:
inicio = time.perf_counter()
sol_bdf = solve_ivp(rhs, (0, 3600), [Ta, Ta, Ta], method="BDF")
duracao = time.perf_counter() - inicio
print(f"BDF: {sol_bdf.t.size - 1} passos, {duracao:.2f} s")
print(f"Tj máxima: {sol_bdf.y[0].max():.2f} °C (exata {T_ex[0].max():.2f});  "
      f"dissipador no fim: {sol_bdf.y[2, -1]:.2f} °C (exata {T_ex[2, -1]:.2f})")

Muito mais rápido, e errado de outro jeito: o dissipador terminou quase 10 K mais frio que o exato. Com passos de vários segundos, o BDF **pula** trocas de carga: um passo que começa e termina em trechos de 30 W não "vê" os 300 W no meio dele, e o módulo recebe menos energia do que devia.

### 6.3 A correção

Um método implícito, **parando em cada troca de carga**. Como $f$ é linear, a jacobiana é a própria matriz $A$, e passá-la ao integrador poupa as avaliações que ele gastaria para estimá-la:

In [ ]:
inicio = time.perf_counter()
t, T = 0.0, np.array([Ta, Ta, Ta])
passos, Tj_max, trechos = 0, Ta, []
while t < 3600 - 1e-9:
    dur = 20.0 if t % 60 < 20 - 1e-9 else 40.0
    meio = t + dur / 2                           # avalia a potência no meio do trecho, longe das bordas
    P = float(potencia(meio))
    s = solve_ivp(lambda tt, TT: A @ TT + np.array([P / Cj, 0.0, Ta / (Rda * Cd)]), (t, t + dur), T,
                  method="Radau", jac=A, rtol=1e-6, atol=1e-6)
    passos += s.t.size - 1
    Tj_max = max(Tj_max, s.y[0].max())
    trechos.append(s)
    t, T = t + dur, s.y[:, -1]
duracao = time.perf_counter() - inicio
Tj_ult = np.concatenate([s.y[0] for s in trechos if s.t[0] >= 3540])
print(f"Radau por trechos: {passos} passos, {duracao:.2f} s")
print(f"Tj máxima: {Tj_max:.2f} °C (exata {T_ex[0].max():.2f});  ΔTj: {np.ptp(Tj_ult):.2f} K "
      f"(exata {np.ptp(T_ex[0][ultimo]):.2f});  dissipador: {T[2]:.2f} °C (exata {T_ex[2, -1]:.2f})")

Agora todos os números batem com a solução exata, com uma fração dos passos do RK45. Os passos são pequenos logo depois de cada troca, quando o chip está mudando, e crescem para segundos quando só o dissipador evolui: é a precisão, e não a estabilidade, que escolhe o passo.

> **Lição.** Desconfie de uma simulação que dá muitos passos de tamanho quase constante, e confira se afrouxar a tolerância muda alguma coisa: se não muda, o problema é **rígido**, e o método deve ser implícito (`Radau`, `BDF` ou `LSODA`), de preferência com a jacobiana. E um passo grande cria um problema novo: ele pode **pular** eventos. Entradas descontínuas (chaveamentos, ciclos de carga, válvulas) pedem integração **por trechos**, ou `max_step`. No pedido ao agente, diga as escalas de tempo do problema e onde estão as descontinuidades.

## 7. Quando falha

### 7.1 Estável, mas errado: Euler implícito num oscilador

Uma massa presa a uma mola, sem atrito, $\ddot x = -x$, oscila para sempre. Os autovalores são $\pm i$, no eixo imaginário. O Euler implícito, estável demais, amortece o que não devia:

In [ ]:
osc = lambda t, u: np.array([u[1], -u[0]])
J_osc = lambda t, u: np.array([[0.0, 1.0], [-1.0, 0.0]])
for metodo in ["euler", "trapezio"]:
    r = implicito(osc, (0, 20 * np.pi), [1.0, 0.0], 0.1, metodo, jac=J_osc)
    print(f"{metodo:8s} implícito, h = 0,1, após 10 oscilações: amplitude = {np.hypot(*r.y[-1]):.4f} (exata 1)")

Após dez oscilações, sobram menos de 5 % da amplitude no Euler implícito: um **amortecimento numérico** que não existe no sistema; o trapézio, cujo $|R|$ é exatamente 1 no eixo imaginário, conserva a amplitude. Numa análise de vibrações, um método muito dissipativo esconde ressonâncias. Por isso a dinâmica estrutural usa métodos como o de Newmark (da família do trapézio), e não o Euler implícito.

### 7.2 O trapézio num problema muito rígido

O trapézio é A-estável, mas não L-estável: para $z \to -\infty$, $R(z) \to -1$. Modos muito rápidos não são amortecidos; trocam de sinal a cada passo:

In [ ]:
f_rig = lambda t, y: -1e4 * (y - np.cos(t))
for metodo in ["trapezio", "euler"]:
    r = implicito(f_rig, (0, 1), [0.0], 0.05, metodo, jac=lambda t, y: np.array([[-1e4]]))
    print(f"{metodo:8s}: y - cos(t) nos 6 primeiros passos:", np.array2string(r.y[1:7, 0] - np.cos(r.t[1:7]), precision=4))

Com o trapézio, o erro inicial (a solução começa em 0, longe de $\cos 0 = 1$) fica oscilando com sinal alternado e mal decai, porque $R(-500) \approx -0{,}992$. O Euler implícito o elimina no primeiro passo. Esse "tremor" do método de Crank-Nicolson reaparece nas EDPs de difusão (capítulo 15), quando a condição inicial tem uma descontinuidade.

### 7.3 Nenhuma tolerância salva uma trajetória caótica

No sistema de Lorenz, o valor de $x(40)$ depende da tolerância, por mais apertada que ela seja:

In [ ]:
for rtol in [1e-6, 1e-9, 1e-12]:
    s = solve_ivp(lorenz, (0, 40), u_l, method="DOP853", rtol=rtol, atol=rtol)
    print(f"rtol = {rtol:.0e}: x(40) = {s.y[0, -1]:+8.3f}")

Num estudo de tolerância como o do capítulo 12, os resultados não convergem. Isso não indica um erro no código: indica que a pergunta ("quanto vale $x(40)$?") está além do horizonte de previsão. A pergunta certa é estatística: a média de $z$, a fração do tempo em cada lóbulo, a distribuição das amplitudes. Essas grandezas convergem.

## Resumo

| Método | Tipo | Estabilidade | Quando usar |
| --- | --- | --- | --- |
| Euler, RK4, RK45 | explícito | região limitada | problemas não rígidos |
| Euler implícito | implícito, ordem 1 | L-estável | rígidos, quando a precisão importa pouco; amortece demais |
| Trapézio (Crank-Nicolson) | implícito, ordem 2 | A-estável, não L-estável | rígidos moderados; oscila com modos muito rápidos |
| BDF (`BDF`, `LSODA`) | passo múltiplo implícito | quase A-estável (ordem ≤ 5) | rígidos de grande porte |
| `Radau` | RK implícito, ordem 5 | L-estável | rígidos com tolerância apertada |
| Verlet | explícito, simplético | eixo imaginário até $\lvert z\rvert = 2$ | sistemas conservativos, simulações longas |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que a região de estabilidade do Euler explícito é o disco $|1 + z| \le 1$ e que a do trapézio é o semiplano $\operatorname{Re} z \le 0$.
2. Para o oscilador $\ddot x = -\omega^2 x$ escrito como sistema, calcule os autovalores e mostre que Euler explícito é **instável para qualquer passo**.
3. Para o módulo de potência, qual seria o maior passo estável do Euler explícito? Quantos passos ele daria em uma hora? E em um ano?
4. Explique por que, num problema rígido, o número de avaliações de $f$ do RK45 quase não depende de `rtol`.

**Verificação (com agente)**

5. Peça a um agente a simulação da cinética de Robertson ($\dot y_1 = -0{,}04y_1 + 10^4y_2y_3$, $\dot y_2 = 0{,}04y_1 - 10^4y_2y_3 - 3\cdot10^7y_2^2$, $\dot y_3 = 3\cdot10^7y_2^2$) até $t = 10^5$. Compare `RK45`, `BDF` e `Radau` em número de avaliações, e verifique que $y_1 + y_2 + y_3 = 1$ se conserva.
6. Peça a um agente para simular a órbita do capítulo 12 por 100 voltas com Verlet, com passo fixo. Verifique a energia. Depois troque por um Verlet com passo adaptativo e repita: o que acontece com a conservação?
7. Peça a um agente para estimar a média de $z$ no sistema de Lorenz ao longo de $t \in [10, 1000]$, com três tolerâncias diferentes. A média converge, embora $x(1000)$ não?

**Projeto aberto**

8. **Ciclo de carga real.** Monte um perfil de potência de um ônibus urbano (paradas a cada 400 m, acelerações, frenagens regenerativas) e simule um dia de operação do módulo. Conte os ciclos de temperatura com o método *rainflow* e estime o dano por fadiga com uma lei do tipo $N_f \propto \Delta T_j^{-5}$. Compare o custo computacional de `RK45`, `LSODA` e `Radau` por trechos.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulo 26.
- E. Hairer e G. Wanner, *Solving Ordinary Differential Equations II: Stiff and Differential-Algebraic Problems*, 2ª ed., Springer, 1996.
- E. Hairer, C. Lubich e G. Wanner, "Geometric numerical integration illustrated by the Störmer-Verlet method", *Acta Numerica*, 12, 2003.
- S. H. Strogatz, *Nonlinear Dynamics and Chaos*, 2ª ed., Westview Press, 2015, capítulo 9 (o sistema de Lorenz).